#### Main Task
- Big picture: The researchers want to measure neighborhood resources around patients (like grocery stores or fitness centers) to study how these resources relate to depression and hypertension over time.
- How this is done:
    - Link businesses from the MAPSCorps dataset to patients using ZIP codes and/or 2020 census tracts (not exact locations).
    - Use business location + type (PlaceType) to understand what resources are available in each neighborhood.
- For each patient visit:
    - Count how many nearby businesses are open/active
    - Group by type of business (aka PlaceType) -- There are 17 different categories total. The final dataset would likely have 17 new columns, one for each business type
- There is one business dataset per year, but we only have the 2019 data

#### My Task 
- The business data only has latitude/longitude
- The patient data cannot use latitude/longitude for privacy reasons
- Patient locations are only allowed to be shared as census tract or ZIP code
- Therefore, the business data has to be converted from lat/long --> census tract before anything can be matched

1) Use the latitude/longitude in the MAPSCorps data to determine the census tract that each business falls in
2) Use the 2020 census tracts (Almost all CAP sites use the 2020 tracts)
3) Make sure each business has at least a valid 5-digit ZIP code and a census tract ID
4) Deliver a cleaned version of the data where each business is assigned to a tract

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Step 1: Read the csv and do quick analysis
chi_data = pd.read_csv("Chicago_Data_2019 - Chicago_Data_2019.csv",low_memory=False)

chi_data.head()

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
0,10 Fortune Nails Spa,4704,N Kimball Ave,NaN,NaN,Albany Park,Chicago,27832,Active,Personal Service,Beauty,No,2019,Beauty Shops,773-293-6065,NaN,POINT(-87.7133731857674 41.9667367834809),-87.71337319,41.96673678,NaN
1,123remodeling,5070,N Kimberly Ave,NaN,NaN,Archer Heights,Chicago,10167,Active,Trade Service,Building trades,No,2019,Sngl-Fam Hsng Cnstr,773-685-6095,NaN,POINT(-87.7437648312545 41.9729256973838),-87.74376483,41.9729257,NaN
2,1st American Insurance,4445,N Pulaski Rd,NaN,NaN,Armour Square,Chicago,282,Active,"Financial, Insurance, Real Estate, Legal, and ...",Insurance agency,No,2019,Insurance Agnts Sv,773-467-2200,NaN,POINT(-87.7278084835625 41.9621980551093),-87.72780848,41.96219806,NaN
3,24-7 Garage Door Services,4010,W Lawrence Ave,NaN,NaN,Ashburn,Chicago,NaN,Out of Business,Trade Service,Building trades,No,2019,Carpentry Work,773-756-3358,NaN,POINT(-87.7284117606663 41.9683008587725),-87.72841176,41.96830086,NaN
4,240 Lounge & Cafe,3948,W Lawrence Ave,NaN,NaN,Auburn Gresham,Chicago,NaN,Active,Arts and Entertainment,Club-lounge-bar,No,2019,Drinking Places,773-267-0474,NaN,POINT(-87.7275671412799 41.96830985887),-87.72756714,41.96830986,NaN


In [4]:
# Check the dtypes
chi_data.dtypes

Name                object
BuildingNumber      object
Street              object
Unit                object
Fraction            object
GeoArea             object
City                object
Zip                 object
PlaceStatus         object
PlaceType           object
PlaceSubType        object
PrivateResidence    object
MappedYear          object
Note                object
PhoneNumber         object
Email               object
Geolocation         object
Lng                 object
Lat                 object
Website             object
dtype: object

In [5]:
# All are objects so let's convert to the correct dtypes for conversions later on
# First, check if there are nulls:
chi_data.isnull().sum()

Name                    0
BuildingNumber          0
Street                  0
Unit                36131
Fraction            39436
GeoArea             39373
City                    0
Zip                 32596
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                21659
PhoneNumber          4397
Email               39275
Geolocation            29
Lng                    30
Lat                    29
Website             38569
dtype: int64

In [6]:
# There are nulls in the longitude and latitude
# Luckily, lat and long for each business is something that can be looked up manually
chi_data[chi_data['Lng'].isnull()]

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
44,Albany Park Fresh Market,3632,W Lawrence Ave,NaN,NaN,Lincoln Square,Chicago,NaN,Out of Business,Retail,Small Grocery,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
109,Atlas Resale Shop,4655,N Elston Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Retail,Resale,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
859,Mobile Document Shreding,4455,N Elston Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Trade Service,"Copy, mail, and print shops",No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1218,Unknown,3520,W Montrose Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Fitness,Weight loss,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1219,Unknown,3520,W Montrose Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Fitness,Weight loss,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2177,Ben's Tire Service,9020,S Halsted St,NaN,NaN,NaN,Chicago,NaN,Out of Business,Trade Service,Vehicle services,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2277,Englewood Realty,7700,S Halsted St,NaN,NaN,NaN,Chicago,NaN,Out of Business,"Financial, Insurance, Real Estate, Legal, and ...",Property service office,No,2019,NaN,773-224-5700,NaN,NaN,NaN,NaN,NaN
4555,Bazar Jasmin,2930,N Milwaukee Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Retail,Clothing,No,2019,NaN,773-517-2787,NaN,NaN,NaN,NaN,NaN
4938,Invitaciones Martinez,2926,N Milwaukee Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,Retail,Gift-art-ethnic,No,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5056,Luggage Warehouse,2874,N Milwaukee Ave,NaN,NaN,NaN,Chicago,NaN,Out of Business,"Wholesale, storage and transit",Warehouse,No,2019,NaN,773-486-4050,NaN,NaN,NaN,NaN,NaN


In [7]:
# We can start manually imputing -- using https://www.mapcoordinates.net
# Use the index and col names to impute

chi_data.loc[44, ['Lat', 'Lng']] = [41.968415, -87.719643]
chi_data.loc[109, ['Lat', 'Lng']] = [41.969301, -87.778015]
chi_data.loc[859, ['Lat', 'Lng']] = [41.961842, -87.730859]
chi_data.loc[1218, ['Lat', 'Lng']] = [41.961172, -87.716348]
chi_data.loc[1219, ['Lat', 'Lng']] = [41.961172, -87.716348]
chi_data.loc[2177, ['Lat', 'Lng']] = [41.730280, -87.643571]
chi_data.loc[2277, ['Lat', 'Lng']] = [41.754148, -87.644243]
chi_data.loc[4555, ['Lat', 'Lng']] = [41.934335, -87.716431]
chi_data.loc[4938, ['Lat', 'Lng']] = [41.934283, -87.716126]
chi_data.loc[5056, ['Lat', 'Lng']] = [41.933428, -87.715031]
chi_data.loc[5813, ['Lat', 'Lng']] = [41.935859, -87.766756]
chi_data.loc[6449, ['Lat', 'Lng']] = [41.9380, -87.7663] # this one is an estimate because the site was not giving me the right coords
chi_data.loc[6798, ['Lat', 'Lng']] = [41.916961, -87.750170]
chi_data.loc[6854, ['Lat', 'Lng']] = [41.916968, -87.749631]
chi_data.loc[7967, ['Lat', 'Lng']] = [41.841013, -87.646242]
chi_data.loc[8574, ['Lat', 'Lng']] = [41.820812, -87.692822]
chi_data.loc[8679, ['Lat', 'Lng']] = [41.814864, -87.703455]
chi_data.loc[8903, ['Lat', 'Lng']] = [41.816265, -87.700526]
chi_data.loc[9139, ['Lat', 'Lng']] = [41.823195, -87.704304]
chi_data.loc[14207, ['Lat', 'Lng']] = [41.751191, -73.880268]
chi_data.loc[14694, ['Lat', 'Lng']] = [41.749150, -87.741891]
chi_data.loc[15318, ['Lat', 'Lng']] = [41.917136, -87.735432]
chi_data.loc[21074, ['Lat', 'Lng']] = [41.920840, -87.694436]
chi_data.loc[21119, ['Lat', 'Lng']] = [41.919220, -87.691830]
chi_data.loc[21414, ['Lat', 'Lng']] = [41.924961, -87.710297]
chi_data.loc[25009, ['Lat', 'Lng']] = [41.885376, -87.644892]
chi_data.loc[25182, ['Lat', 'Lng']] = [41.882968, -87.663191]
chi_data.loc[25754, ['Lat', 'Lng']] = [41.883215, -87.648179]
chi_data.loc[26422, ['Lat', 'Lng']] = [41.873657, -87.682616]
chi_data.loc[36690, ['Lat', 'Lng']] = [41.710633, -87.649319]

# Although most of these businesses are out of business, it's best to impute now to avoid issues later on

In [8]:
# Adjust Geolocation accordingly

# Ensure coordinates are numeric
chi_data['Lat'] = pd.to_numeric(chi_data['Lat'], errors='coerce')
chi_data['Lng'] = pd.to_numeric(chi_data['Lng'], errors='coerce')

# Fill Geolocation only where it's currently null
mask = chi_data['Geolocation'].isnull()
chi_data.loc[mask, 'Geolocation'] = 'POINT(' + chi_data.loc[mask, 'Lng'].astype(str) + ' ' + chi_data.loc[mask, 'Lat'].astype(str) + ')'


In [9]:
# Check
chi_data.loc[36690, :]

Name                                             Granny Goose Daycare
BuildingNumber                                                  10056
Street                                                 S Carpenter St
Unit                                                              NaN
Fraction                                                          NaN
GeoArea                                                           NaN
City                                                          Chicago
Zip                                                               NaN
PlaceStatus                                           Out of Business
PlaceType                                       Childcare and Schools
PlaceSubType                                             Home daycare
PrivateResidence                                                  Yes
MappedYear                                                       2019
Note                Called (773) 239-6095 and spoke with owner.  O...
PhoneNumber         

In [10]:
# Let's check the imputed numbers
chi_data.isnull().sum()

Name                    0
BuildingNumber          0
Street                  0
Unit                36131
Fraction            39436
GeoArea             39373
City                    0
Zip                 32596
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                21659
PhoneNumber          4397
Email               39275
Geolocation             0
Lng                     2
Lat                     2
Website             38569
dtype: int64

In [11]:
chi_data.loc[chi_data['Lat'].isnull()]
# no adjustment here...we can drop these observations because they are repeats of the col names


,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
13841,Name,BuildingNumber,Street,Unit,Fraction,NaN,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,NaN,NaN,Website
13842,Name,BuildingNumber,Street,Unit,Fraction,NaN,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,NaN,NaN,Website


In [12]:
chi_data = chi_data.dropna(subset=['Lat', 'Lng'])
chi_data.isnull().sum()

Name                    0
BuildingNumber          0
Street                  0
Unit                36131
Fraction            39436
GeoArea             39371
City                    0
Zip                 32596
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                21659
PhoneNumber          4397
Email               39275
Geolocation             0
Lng                     0
Lat                     0
Website             38569
dtype: int64

In [13]:
# Imputing is done! Now we can start to convert the lat and long to census tract
# We can do this using Geocodio API
    # https://www.geocod.io/docs/?python#introduction
    # a RESTful API designed for forward/reverse geocoding of US and Canadian addresses --> can convert long/lat to census tracts

In [14]:
pip install geocodio-library-python

Note: you may need to restart the kernel to use updated packages.


In [15]:
# Test using the example from Geocod.io

from geocodio import Geocodio

client = Geocodio("53e709590b05ed6376c9e96b7c739595cb7e57c")

response = client.reverse(
    "38.886672,-77.094735",
    fields=["census2020", "census"]
)

result = response.results[0]
census_data = result.fields._census

tract_code_2020 = census_data['census2020'].tract_code

print("Census 2020 tract code:", tract_code_2020)

Census 2020 tract code: 101801


In [16]:
# Now we now we can get the tract code using coords!
# Let's try doing it for one of our rows (row 44)
# Also, this uses the 2020 census

response = client.reverse(
    "41.968415, -87.719643",
    fields=["census2020", "census"]
)

result = response.results[0]
census_data = result.fields._census

tract_code_2020 = census_data['census2020'].tract_code

print("Census 2020 tract code:", tract_code_2020)

Census 2020 tract code: 140301


In [17]:
# This census tract code is correct! (Checked using a quick search)
# Let's now do this (vectorized) for our dataframe

coords = chi_data[['Lat', 'Lng']].apply(tuple, axis=1).tolist()

responses = client.reverse(coords, fields=["census2020", "census"])

tract_codes = []
for result in responses:
    if result.fields._census and 'census2020' in result.fields._census:
        tract_codes.append(result.fields._census['census2020'].tract_code)
    else:
        tract_codes.append(None)

# Add tract codes to DataFrame
chi_data['tract_code_2020'] = tract_codes



Error response: 403 - {"error":"You have exceeded the free tier. Please add a payment method or check that you are using the intended API key.","reference":"https:\/\/www.geocod.io\/geocodio-403-forbidden-error\/"}


AuthenticationError: {"error":"You have exceeded the free tier. Please add a payment method or check that you are using the intended API key.","reference":"https:\/\/www.geocod.io\/geocodio-403-forbidden-error\/"}

In [ ]:
chi_data.shape

In [ ]:
# Let's try using the US Census Bureau API geocoder instead of Geocod.io

# This site only allows geocoding for 10,000 records at a time
# Must cut the csv in 4 if we want this to work

import pandas as pd
import numpy as np

# Split into 4 parts
parts = np.array_split(chi_data, 4)

# Save each part
for i, part in enumerate(parts):
    part.to_csv(f'split_file_{i+1}.csv', index=False)


In [21]:
# Let's try finding the tract for sample coords

import requests

lon = -87.6298
lat = 41.8781

url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates" # url for the geocoder

params = {
    "x": lon,
    "y": lat,
    "benchmark": "Public_AR_Current",
    "vintage": "Census2020_Current", # using the 2020 census
    "format": "json"
}

response = requests.get(url, params=params) # requesting a json file with the records we are looking for using url and params
data = response.json()

tract_geoid = data["result"]["geographies"]["Census Tracts"][0]["GEOID"] # searching the json file to get the information we want
print(tract_geoid)


17031839100


In [22]:
# Define a function for geocoding using the principles from the cell above

def get_tract(Lat, Lng):
    params = {
        "x": Lng,
        "y": Lat,
        "benchmark": "Public_AR_Current",
        "vintage": "Census2020_Current",
        "format": "json"
    }
    r = requests.get(url, params=params)
    try:
        return r.json()["result"]["geographies"]["Census Tracts"][0]["GEOID"]
    except (KeyError, IndexError):
        return None

# Read the split 
split_1 = pd.read_csv('split_file_1.csv') # we made the four splits but haven't read them yet

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_1[["Lat", "Lng"]].drop_duplicates() # drop any duplicates that may elongate the runtime
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor: # sends in 8 API requests at once instead of doing it one at a time
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])), # uses var names from the data 
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

# adding in geoid records into the first split
split_1["tract_geoid"] = split_1.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)


In [23]:
split_1.head()
# it worked! took around 5 minutes to run for the split

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,...,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website,tract_geoid
0,10 Fortune Nails Spa,4704,N Kimball Ave,NaN,NaN,Albany Park,Chicago,27832.0,Active,Personal Service,...,No,2019,Beauty Shops,773-293-6065,NaN,POINT(-87.7133731857674 41.9667367834809),-87.713373,41.966737,NaN,17031140701
1,123remodeling,5070,N Kimberly Ave,NaN,NaN,Archer Heights,Chicago,10167.0,Active,Trade Service,...,No,2019,Sngl-Fam Hsng Cnstr,773-685-6095,NaN,POINT(-87.7437648312545 41.9729256973838),-87.743765,41.972926,NaN,17031140400
2,1st American Insurance,4445,N Pulaski Rd,NaN,NaN,Armour Square,Chicago,282.0,Active,"Financial, Insurance, Real Estate, Legal, and ...",...,No,2019,Insurance Agnts Sv,773-467-2200,NaN,POINT(-87.7278084835625 41.9621980551093),-87.727808,41.962198,NaN,17031140601
3,24-7 Garage Door Services,4010,W Lawrence Ave,NaN,NaN,Ashburn,Chicago,NaN,Out of Business,Trade Service,...,No,2019,Carpentry Work,773-756-3358,NaN,POINT(-87.7284117606663 41.9683008587725),-87.728412,41.968301,NaN,17031140400
4,240 Lounge & Cafe,3948,W Lawrence Ave,NaN,NaN,Auburn Gresham,Chicago,NaN,Active,Arts and Entertainment,...,No,2019,Drinking Places,773-267-0474,NaN,POINT(-87.7275671412799 41.96830985887),-87.727567,41.968310,NaN,17031140301


In [24]:
# Doing the same for split 2

split_2 = pd.read_csv('split_file_2.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2["tract_geoid"] = split_2.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [25]:
# Doing split 3

split_3 = pd.read_csv('split_file_3.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_3[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_3["tract_geoid"] = split_3.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [26]:
# Doing split 4

split_4 = pd.read_csv('split_file_4.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_4[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_4["tract_geoid"] = split_4.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [27]:
split_4.head()

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,...,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website,tract_geoid
0,Columbia Marketing & Funding Corporation,3445,N Central Ave,NaN,NaN,NaN,Chicago,NaN,Active,"Financial, Insurance, Real Estate, Legal, and ...",...,Yes,2019,Prsnl Crdt Instutns,708-669-5500,NaN,POINT(-87.7666264899081 41.9435403174205),-87.766626,41.943540,NaN,17031151100
1,Communications Unlisted,5013,W Irving Park Rd,NaN,NaN,NaN,Chicago,NaN,Active,Retail,...,No,2019,NaN,773-282-6222,NaN,POINT(-87.7528390813504 41.9533224955351),-87.752839,41.953322,NaN,17031150800
2,Community First Medical Center,5645,W Addison St,NaN,NaN,NaN,Chicago,NaN,Active,Health Services,...,No,2019,Gnl Mdl Srgl Hsptl,773-282-7000,NaN,POINT(-87.7690035004218 41.9458314885079),-87.769003,41.945831,NaN,17031151200
3,Community First Medical Center Gift Gallery,5645,W Addison St,NaN,NaN,NaN,Chicago,NaN,Active,Retail,...,No,2019,Gft Nvlty Svenr S,773-794-7661,NaN,POINT(-87.7690035004218 41.9458314885079),-87.769003,41.945831,NaN,17031151200
4,Community Savings Bank,4824,W Belmont Ave,NaN,NaN,NaN,Chicago,NaN,Active,"Financial, Insurance, Real Estate, Legal, and ...",...,No,2019,NaN,773-685-5300,NaN,POINT(-87.7480015493408 41.9388876341004),-87.748002,41.938888,NaN,17031151002


In [36]:
# concatenate the splits
chi_data_NEW = pd.concat([split_1,split_2,split_3,split_4])
chi_data_NEW

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,...,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website,tract_geoid
0,10 Fortune Nails Spa,4704,N Kimball Ave,NaN,NaN,Albany Park,Chicago,27832.0,Active,Personal Service,...,No,2019,Beauty Shops,773-293-6065,NaN,POINT(-87.7133731857674 41.9667367834809),-87.713373,41.966737,NaN,17031140701
1,123remodeling,5070,N Kimberly Ave,NaN,NaN,Archer Heights,Chicago,10167.0,Active,Trade Service,...,No,2019,Sngl-Fam Hsng Cnstr,773-685-6095,NaN,POINT(-87.7437648312545 41.9729256973838),-87.743765,41.972926,NaN,17031140400
2,1st American Insurance,4445,N Pulaski Rd,NaN,NaN,Armour Square,Chicago,282.0,Active,"Financial, Insurance, Real Estate, Legal, and ...",...,No,2019,Insurance Agnts Sv,773-467-2200,NaN,POINT(-87.7278084835625 41.9621980551093),-87.727808,41.962198,NaN,17031140601
3,24-7 Garage Door Services,4010,W Lawrence Ave,NaN,NaN,Ashburn,Chicago,NaN,Out of Business,Trade Service,...,No,2019,Carpentry Work,773-756-3358,NaN,POINT(-87.7284117606663 41.9683008587725),-87.728412,41.968301,NaN,17031140400
4,240 Lounge & Cafe,3948,W Lawrence Ave,NaN,NaN,Auburn Gresham,Chicago,NaN,Active,Arts and Entertainment,...,No,2019,Drinking Places,773-267-0474,NaN,POINT(-87.7275671412799 41.96830985887),-87.727567,41.968310,NaN,17031140301
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9857,Woodlawn Community Development Corporation,1500,E 63rd St,NaN,NaN,NaN,Chicago,60637.0,Out of Business,Social Services & Political Advocacy,...,No,2019,NaN,NaN,NaN,POINT(-87.5888051 41.7807726),-87.588805,41.780773,NaN,17031420100
9858,Woodlawn Development Associates,6224,S Kimbark Ave,NaN,NaN,NaN,Chicago,NaN,Active,Social Services & Political Advocacy,...,No,2019,Sngl-Fam Hsng Cnstr,773-643-7495,NaN,POINT(-87.595121 41.781553),-87.595121,41.781553,NaN,17031420200
9859,Woodlawn's Inspired Community Garden,848,E 64th St,NaN,NaN,NaN,Chicago,NaN,Active,Other,...,No,2019,NaN,214-673-7233,NaN,POINT(-87.6039202 41.7788568),-87.603920,41.778857,NaN,17031420800
9860,Write Works Ii Communications,6221,S Kenwood Ave,Apt 1f,NaN,NaN,Chicago,NaN,Unknown,"Financial, Insurance, Real Estate, Legal, and ...",...,Yes,2019,NaN,773-643-7433,NaN,POINT(-87.592812 41.7816987),-87.592812,41.781699,NaN,17031420200


In [32]:
# the concatenated csv has 39450 rows and 21 cols; let's check if this matches the original csv
chi_data.shape

(39450, 20)

In [35]:
#yes, they match! this means that all of the rows were geocoded correctly and an extra column was added for the geoid
chi_data.tail()

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
39447,Woodlawn Community Development Corporation,1500,E 63rd St,NaN,NaN,NaN,Chicago,60637,Out of Business,Social Services & Political Advocacy,Social service,No,2019,NaN,NaN,NaN,POINT(-87.5888051 41.7807726),-87.588805,41.780773,NaN
39448,Woodlawn Development Associates,6224,S Kimbark Ave,NaN,NaN,NaN,Chicago,NaN,Active,Social Services & Political Advocacy,Advocacy,No,2019,Sngl-Fam Hsng Cnstr,773-643-7495,NaN,POINT(-87.595121 41.781553),-87.595121,41.781553,NaN
39449,Woodlawn's Inspired Community Garden,848,E 64th St,NaN,NaN,NaN,Chicago,NaN,Active,Other,Community garden,No,2019,NaN,214-673-7233,NaN,POINT(-87.6039202 41.7788568),-87.603920,41.778857,NaN
39450,Write Works Ii Communications,6221,S Kenwood Ave,Apt 1f,NaN,NaN,Chicago,NaN,Unknown,"Financial, Insurance, Real Estate, Legal, and ...",Other professional,Yes,2019,NaN,773-643-7433,NaN,POINT(-87.592812 41.7816987),-87.592812,41.781699,NaN
39451,Y2kwanzaa Org,6616,S Minerva Ave,NaN,NaN,NaN,Chicago,60637,Out of Business,Unknown,Unknown,Yes,2019,NaN,NaN,NaN,POINT(-87.5975532 41.7744971),-87.597553,41.774497,NaN


In [37]:
# It also appears that they are in the same order!

#### What we have done so far:

- Cleaned the original chi_data csv by checking any null values for latitutde and longitude
- Manually imputed the cooordinates for these; there may be a more automated way to do this but it didn't take too long for only 30 values
- Tried to use Geocod.io to geocode the tract codes
    - Had issues with the amount of requests that we could send in
- Use the US census geocoder --> accepts 10,000 records at a time
    - Had to split the original data in 4 to ensure there were no issues with sending in records
        - Each split was a bit less than 10,000
    - Used https://docs.python.org/3/library/concurrent.futures.html to launch the tasks in parallel; the original code took too long because there were too many requests being sent in
    - Essentially went into the geocoder site with different routes to return a json record with all of the information with the lat and long inputs given
    - This was applied to each observation in the csv to get tract codes for each long/lat coordinate
    - Concatenated the split csv files to get same size of the original data
    - Checked with the original csv file to see if they were of the same size/order
    - Also manually checked the tract code for a couple of randomly selected observations to check accuracy

#### Next Step:
- Do the same but for zip codes now...some values already have zip included but must now impute the nulls
- We're going to do this using the 2020 ZCTA TIGER/Line Shapefiles: https://www.census.gov/programs-surveys/geography/guidance/geo-areas/zctas.html
- Also using the geopandas package

In [63]:
# Let's check what the current zip codes look like right now

chi_data_NEW['Zip'].unique()

array([27832., 10167.,   282.,    nan, 60618., 60630., 60625., 60632.,
       60616., 60620., 60652., 60651., 60644., 60707., 60639., 60612.,
       60302., 60617., 60619., 60641., 60629., 60634., 60643., 60608.,
       60609., 60633., 60827., 60636., 60638., 60653., 60624., 60621.,
       60637., 60615., 60647., 60622., 60607., 60614., 60628., 60661.,
       60606., 60623., 60649., 60640., 60642., 60654.])

In [64]:
# Most of these zip codes look fine, however, there are some that don't look normal. 
# For example, 27832 is the zip code for north carolina, not chicago
# We can't just isolate the null values from the data frame, it may just be better to make a new column

In [38]:
pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 4.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 3.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 7.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [pyogrio]

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [53]:
# Remember, chi_data_NEW is the newly updated df with the tract codes
# We also have split_file(s) if there are issues with too many records being geocoded at once

import geopandas as gpd
from shapely.geometry import Point

gdf_points = gpd.GeoDataFrame(
    chi_data_NEW,
    geometry=gpd.points_from_xy(chi_data_NEW["Lng"], chi_data_NEW["Lat"]),
    crs="EPSG:4326"  # WGS84 (standard lat/lon)
)

# spatial joins only work on geometry objects

In [65]:
zcta = gpd.read_file('tl_2020_us_zcta520.shp')

zcta.columns #check the columns that are in the shape file

Index(['ZCTA5CE20', 'GEOID20', 'CLASSFP20', 'MTFCC20', 'FUNCSTAT20', 'ALAND20',
       'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'geometry'],
      dtype='object')

In [92]:
zcta = zcta.to_crs(gdf_points.crs)

gdf_with_zip = gpd.sjoin(
    gdf_points,
    zcta[["ZCTA5CE20", "geometry"]],
    how="left",
    predicate="within"
)

chi_data_FINAL = gdf_with_zip.drop(
    columns=["geometry", "index_right"]
)


In [93]:
chi_data_FINAL

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,...,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website,tract_geoid,ZCTA5CE20
0,10 Fortune Nails Spa,4704,N Kimball Ave,NaN,NaN,Albany Park,Chicago,27832.0,Active,Personal Service,...,2019,Beauty Shops,773-293-6065,NaN,POINT(-87.7133731857674 41.9667367834809),-87.713373,41.966737,NaN,17031140701,60625
1,123remodeling,5070,N Kimberly Ave,NaN,NaN,Archer Heights,Chicago,10167.0,Active,Trade Service,...,2019,Sngl-Fam Hsng Cnstr,773-685-6095,NaN,POINT(-87.7437648312545 41.9729256973838),-87.743765,41.972926,NaN,17031140400,60630
2,1st American Insurance,4445,N Pulaski Rd,NaN,NaN,Armour Square,Chicago,282.0,Active,"Financial, Insurance, Real Estate, Legal, and ...",...,2019,Insurance Agnts Sv,773-467-2200,NaN,POINT(-87.7278084835625 41.9621980551093),-87.727808,41.962198,NaN,17031140601,60625
3,24-7 Garage Door Services,4010,W Lawrence Ave,NaN,NaN,Ashburn,Chicago,NaN,Out of Business,Trade Service,...,2019,Carpentry Work,773-756-3358,NaN,POINT(-87.7284117606663 41.9683008587725),-87.728412,41.968301,NaN,17031140400,60630
4,240 Lounge & Cafe,3948,W Lawrence Ave,NaN,NaN,Auburn Gresham,Chicago,NaN,Active,Arts and Entertainment,...,2019,Drinking Places,773-267-0474,NaN,POINT(-87.7275671412799 41.96830985887),-87.727567,41.968310,NaN,17031140301,60625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9857,Woodlawn Community Development Corporation,1500,E 63rd St,NaN,NaN,NaN,Chicago,60637.0,Out of Business,Social Services & Political Advocacy,...,2019,NaN,NaN,NaN,POINT(-87.5888051 41.7807726),-87.588805,41.780773,NaN,17031420100,60637
9858,Woodlawn Development Associates,6224,S Kimbark Ave,NaN,NaN,NaN,Chicago,NaN,Active,Social Services & Political Advocacy,...,2019,Sngl-Fam Hsng Cnstr,773-643-7495,NaN,POINT(-87.595121 41.781553),-87.595121,41.781553,NaN,17031420200,60637
9859,Woodlawn's Inspired Community Garden,848,E 64th St,NaN,NaN,NaN,Chicago,NaN,Active,Other,...,2019,NaN,214-673-7233,NaN,POINT(-87.6039202 41.7788568),-87.603920,41.778857,NaN,17031420800,60637
9860,Write Works Ii Communications,6221,S Kenwood Ave,Apt 1f,NaN,NaN,Chicago,NaN,Unknown,"Financial, Insurance, Real Estate, Legal, and ...",...,2019,NaN,773-643-7433,NaN,POINT(-87.592812 41.7816987),-87.592812,41.781699,NaN,17031420200,60637


#### Aggregating the DataFrame Outputs:
- We want one output with census_tract, PlaceType, count_of_Active_PlaceStatus
- We want another output with ZCTA_Zipcode, PlaceType, count_of_Active_PlaceStatus

In [76]:
# Must filter out the observations only that are "Acitve" for the PlaceStatus var

chi_data_active = chi_data_FINAL.loc[chi_data_FINAL['PlaceStatus'] == 'Active']

In [83]:
# This is the df for census_tract, PlaceType, count_of_Active_PlaceStatus

tracts_active_df = (
    chi_data_active
    .groupby(["tract_geoid", "PlaceType"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

tracts_active_df = tracts_active_df.rename(columns = {'tract_geoid':'census_tract'})

tracts_active_df = tracts_active_df.sort_values(["census_tract", "PlaceType"])

tracts_active_df

,census_tract,PlaceType,count_of_Active_PlaceStatus
0,17031031000,Arts and Entertainment,8
1,17031031000,Childcare and Schools,1
2,17031031000,Dining,9
3,17031031000,"Financial, Insurance, Real Estate, Legal, and ...",22
4,17031031000,Health Services,27
...,...,...,...
5606,17031844700,Service or Programmed Residential Space,1
5607,17031844700,Social Services & Political Advocacy,1
5608,17031844700,Trade Service,4
5609,17031980100,"Financial, Insurance, Real Estate, Legal, and ...",1


In [85]:
# Save it as a csv
tracts_active_df.to_csv('MAPSCorps_2019_Tract_PlaceType_Counts.csv',index = False)

In [94]:
# Do the same for zip codes

zip_active_df = (
    chi_data_active
    .groupby(["ZCTA5CE20", "PlaceType"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

zip_active_df = zip_active_df.rename(columns = {'ZCTA5CE20':'ZCTA_Zipcode'})

zip_active_df = zip_active_df.sort_values(["ZCTA_Zipcode", "PlaceType"])

zip_active_df

,ZCTA_Zipcode,PlaceType,count_of_Active_PlaceStatus
0,60131,Trade Service,1
1,60302,Dining,1
2,60302,Retail,1
3,60402,Trade Service,1
4,60419,Religious (except school or residence),1
...,...,...,...
638,60827,Retail,5
639,60827,Service or Programmed Residential Space,1
640,60827,Social Services & Political Advocacy,2
641,60827,Trade Service,3


In [98]:
# Save it as a csv
zip_active_df.to_csv('MAPSCorps_2019_Zip_PlaceType_Counts.csv',index = False)

In [99]:
# Save the fully transformed df to a csv
chi_data_FINAL.to_csv('MAPSCorps_2019_Converted.csv',index = False)